# 🔧 Notebook 03 — Hyperparameter Tuning
## TourMate AI · Tourism Recommendation System

> **Objective:** Systematically tune every model and recommendation component using rigorous cross-validation.  
> Document before/after improvements, explain every parameter, and save best configurations.

| Component | Method | Metric |
|---|---|---|
| Classification (RF, ET, GB, XGB, LGB) | RandomizedSearchCV | F1-Weighted, StratifiedKFold-5 |
| Regression (RF, ET, GB, XGB, LGB) | RandomizedSearchCV | R², KFold-5 |
| TF-IDF Vectorizer | Grid search | Coverage × Similarity |
| NearestNeighbors | Grid search | Recall@K |
| Hybrid Weights | Ablation study | Quality × Diversity × Novelty |
| KMeans Clustering | K-selection | Silhouette Score |

---

In [ ]:
# ─── 0. SETUP ──────────────────────────────────────────────────────────────────
import sys, os, json, time, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import (RandomizedSearchCV, GridSearchCV,
    StratifiedKFold, KFold, cross_val_score, learning_curve)
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, GradientBoostingRegressor,
    RandomForestRegressor, ExtraTreesRegressor)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
    mean_squared_error, r2_score, mean_absolute_error,
    precision_score, recall_score, silhouette_score)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import randint, uniform, loguniform
import scipy.stats as scipy_stats

PALETTE = {'sky':'#0ea5e9','sky_d':'#0284c7','sky_l':'#e0f2fe',
           'mint':'#52c27a','sand':'#d4a373','snow':'#f1f5f9','muted':'#64748b'}
CHART_COLORS = ['#0ea5e9','#52c27a','#d4a373','#f97316','#8b5cf6','#ec4899']
plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False})

# Optional boosters
try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGB = True; print("✓ XGBoost available")
except ImportError:
    HAS_XGB = False; print("⚠ XGBoost not available")
try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    HAS_LGB = True; print("✓ LightGBM available")
except ImportError:
    HAS_LGB = False; print("⚠ LightGBM not available")

print("✓ Setup complete")


In [ ]:
# ─── 1. DATA PREPARATION ──────────────────────────────────────────────────────
from src.data_preprocessing import load_dataset, clean_dataset, build_attraction_catalog
from src.recommender import ContentBasedRecommender, HybridRecommender

df = load_dataset('../data/tourism_recommendation_dataset_en.csv')
df = clean_dataset(df)
catalog = build_attraction_catalog(df)

# Feature engineering
enc_cols = ['attraction_category','season','gender','age_group',
            'attraction_level','province']
encoders = {}
for col in enc_cols:
    le = LabelEncoder()
    df[col+'_enc'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

FEATURE_COLS = ['age','ticket_price','visit_duration_hours','spend_amount',
                'other_spend','trip_days','is_holiday_enc','is_group_tour_enc',
                'attraction_category_enc','season_enc','gender_enc',
                'age_group_enc','attraction_level_enc','province_enc']
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

le_target = LabelEncoder()
y_clf = le_target.fit_transform(df['recommendation_level'].astype(str))
y_reg = df['rating'].values
X     = df[FEATURE_COLS].fillna(0)

X_train_c, X_test_c, y_train_c, y_test_c =     __import__('sklearn.model_selection', fromlist=['train_test_split']).train_test_split(
        X, y_clf, test_size=0.20, random_state=42, stratify=y_clf)
X_train_r, X_test_r, y_train_r, y_test_r =     __import__('sklearn.model_selection', fromlist=['train_test_split']).train_test_split(
        X, y_reg, test_size=0.20, random_state=42)

print(f"Train: {X_train_c.shape[0]:,} | Test: {X_test_c.shape[0]:,}")
print(f"Features: {len(FEATURE_COLS)}")
print(f"Target classes: {le_target.classes_}")


## 1. Classification Hyperparameter Tuning

In [ ]:
# ─── 1.1 BEFORE TUNING: BASELINE SCORES ──────────────────────────────────────
# Establish clear before/after comparison baselines
print("=" * 65)
print("BASELINE (Default Parameters) — Before Tuning")
print("=" * 65)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baselines = {}
baseline_models = {
    'Random Forest':     RandomForestClassifier(n_estimators=100, random_state=42),
    'Extra Trees':       ExtraTreesClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
}
if HAS_XGB:
    baseline_models['XGBoost'] = XGBClassifier(n_estimators=100, random_state=42,
                                                 eval_metric='mlogloss', verbosity=0)
if HAS_LGB:
    baseline_models['LightGBM'] = LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)

print(f"\n{'Model':<22} {'CV F1-W Mean':>13} {'CV F1-W Std':>12} {'Test F1-W':>10}")
print("─" * 62)
for name, model in baseline_models.items():
    t0 = time.time()
    cv = cross_val_score(model, X, y_clf, cv=skf, scoring='f1_weighted', n_jobs=-1)
    model.fit(X_train_c, y_train_c)
    test_f1 = f1_score(y_test_c, model.predict(X_test_c), average='weighted')
    elapsed = time.time() - t0
    baselines[name] = {'cv_mean': cv.mean(), 'cv_std': cv.std(), 'test_f1': test_f1}
    print(f"  {name:<20} {cv.mean():>13.4f} {cv.std():>12.4f} {test_f1:>10.4f}  ({elapsed:.1f}s)")


In [ ]:
# ─── 1.2 RANDOM FOREST — RANDOMIZED SEARCH ───────────────────────────────────
rf_param_dist = {
    'n_estimators':      randint(100, 500),
    'max_depth':         [8, 10, 12, 15, 20, None],
    'min_samples_split': randint(2, 15),
    'min_samples_leaf':  randint(1, 8),
    'max_features':      ['sqrt', 'log2', 0.3, 0.5],
    'class_weight':      ['balanced', 'balanced_subsample', None],
    'criterion':         ['gini', 'entropy'],
    'bootstrap':         [True, False],
}

print("Tuning Random Forest (RandomizedSearchCV, 5-fold, n_iter=30)...")
print("Parameter space:")
for param, space in rf_param_dist.items():
    print(f"  {param:<22}: {space}")
print()

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_param_dist, n_iter=30, cv=skf,
    scoring='f1_weighted', random_state=42,
    n_jobs=-1, verbose=0, refit=True
)
t0 = time.time()
rf_search.fit(X, y_clf)
elapsed = time.time() - t0

print(f"\n✓ Search complete in {elapsed:.1f}s")
print(f"  Best CV F1-Weighted : {rf_search.best_score_:.4f}")
print(f"  Best Parameters     :")
for k, v in rf_search.best_params_.items():
    print(f"    {k:<22}: {v}")

# Test set score
rf_best = rf_search.best_estimator_
rf_test_f1 = f1_score(y_test_c, rf_best.predict(X_test_c), average='weighted')
rf_baseline_f1 = baselines['Random Forest']['test_f1']
improvement_rf = rf_test_f1 - rf_baseline_f1
print(f"\n  Before tuning Test F1: {rf_baseline_f1:.4f}")
print(f"  After  tuning Test F1: {rf_test_f1:.4f}")
print(f"  Improvement          : {improvement_rf:+.4f} ({improvement_rf/rf_baseline_f1*100:+.2f}%)")


In [ ]:
# ─── 1.3 GRADIENT BOOSTING — RANDOMIZED SEARCH ───────────────────────────────
gb_param_dist = {
    'n_estimators':    randint(100, 400),
    'max_depth':       [3, 4, 5, 6, 7],
    'learning_rate':   loguniform(0.01, 0.3),
    'subsample':       uniform(0.6, 0.4),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf':  randint(1, 10),
    'max_features':    ['sqrt', 'log2', None],
    'loss':            ['log_loss'],
}

print("Tuning Gradient Boosting (RandomizedSearchCV, 5-fold, n_iter=25)...")
gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_param_dist, n_iter=25, cv=skf,
    scoring='f1_weighted', random_state=42,
    n_jobs=-1, verbose=0, refit=True
)
t0 = time.time()
gb_search.fit(X, y_clf)
elapsed = time.time() - t0

print(f"✓ Search complete in {elapsed:.1f}s")
print(f"  Best CV F1-Weighted : {gb_search.best_score_:.4f}")
print(f"  Best Parameters     :")
for k, v in gb_search.best_params_.items():
    print(f"    {k:<22}: {v:.4f}" if isinstance(v, float) else f"    {k:<22}: {v}")

gb_best = gb_search.best_estimator_
gb_test_f1 = f1_score(y_test_c, gb_best.predict(X_test_c), average='weighted')
gb_baseline = baselines['Gradient Boosting']['test_f1']
print(f"\n  Before → After: {gb_baseline:.4f} → {gb_test_f1:.4f} ({gb_test_f1-gb_baseline:+.4f})")


In [ ]:
# ─── 1.4 XGBOOST TUNING (if available) ───────────────────────────────────────
if HAS_XGB:
    from xgboost import XGBClassifier
    xgb_param_dist = {
        'n_estimators':    randint(100, 400),
        'max_depth':       randint(3, 10),
        'learning_rate':   loguniform(0.01, 0.3),
        'subsample':       uniform(0.6, 0.4),
        'colsample_bytree':uniform(0.5, 0.5),
        'gamma':           uniform(0, 0.5),
        'reg_alpha':       loguniform(0.001, 1.0),
        'reg_lambda':      loguniform(0.1, 10.0),
        'min_child_weight':randint(1, 10),
    }
    print("Tuning XGBoost (n_iter=25)...")
    xgb_search = RandomizedSearchCV(
        XGBClassifier(use_label_encoder=False, eval_metric='mlogloss',
                       random_state=42, verbosity=0, n_jobs=-1),
        xgb_param_dist, n_iter=25, cv=skf,
        scoring='f1_weighted', random_state=42, n_jobs=-1
    )
    xgb_search.fit(X, y_clf)
    xgb_best = xgb_search.best_estimator_
    xgb_f1 = f1_score(y_test_c, xgb_best.predict(X_test_c), average='weighted')
    print(f"  Best CV F1: {xgb_search.best_score_:.4f}")
    print(f"  Best Params: {xgb_search.best_params_}")
    print(f"  Test F1: {xgb_f1:.4f}  (baseline: {baselines.get('XGBoost',{}).get('test_f1',0):.4f})")
else:
    print("XGBoost not available — skipping")
    xgb_best, xgb_f1 = None, 0


In [ ]:
# ─── 1.5 LIGHTGBM TUNING (if available) ──────────────────────────────────────
if HAS_LGB:
    from lightgbm import LGBMClassifier
    lgb_param_dist = {
        'n_estimators':   randint(100, 400),
        'num_leaves':     randint(20, 150),
        'learning_rate':  loguniform(0.01, 0.3),
        'subsample':      uniform(0.6, 0.4),
        'colsample_bytree':uniform(0.5, 0.5),
        'reg_alpha':      loguniform(0.001, 1.0),
        'reg_lambda':     loguniform(0.001, 1.0),
        'min_child_samples': randint(5, 50),
        'max_depth':      [5, 8, 10, 15, -1],
    }
    print("Tuning LightGBM (n_iter=25)...")
    lgb_search = RandomizedSearchCV(
        LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1, n_jobs=-1),
        lgb_param_dist, n_iter=25, cv=skf,
        scoring='f1_weighted', random_state=42, n_jobs=-1
    )
    lgb_search.fit(X, y_clf)
    lgb_best = lgb_search.best_estimator_
    lgb_f1 = f1_score(y_test_c, lgb_best.predict(X_test_c), average='weighted')
    print(f"  Best CV F1: {lgb_search.best_score_:.4f}")
    print(f"  Best Params: {lgb_search.best_params_}")
    print(f"  Test F1: {lgb_f1:.4f}")
else:
    print("LightGBM not available — skipping")
    lgb_best, lgb_f1 = None, 0


In [ ]:
# ─── 1.6 CLASSIFICATION: BEFORE vs AFTER COMPARISON ─────────────────────────
tuned_results = {
    'Random Forest':     rf_test_f1,
    'Gradient Boosting': gb_test_f1,
}
if HAS_XGB and xgb_best: tuned_results['XGBoost'] = xgb_f1
if HAS_LGB and lgb_best: tuned_results['LightGBM'] = lgb_f1

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Before vs After bar chart
models_shown = [m for m in tuned_results if m in baselines]
before_scores = [baselines[m]['test_f1'] for m in models_shown]
after_scores  = [tuned_results[m] for m in models_shown]
x = np.arange(len(models_shown))
width = 0.38

bars1 = axes[0].bar(x - width/2, before_scores, width, label='Before Tuning',
                     color=PALETTE['sand'], alpha=0.85, edgecolor='white')
bars2 = axes[0].bar(x + width/2, after_scores,  width, label='After Tuning',
                     color=PALETTE['sky'],  alpha=0.85, edgecolor='white')
for bar, v in zip(bars1, before_scores):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.002,
                 f'{v:.4f}', ha='center', va='bottom', fontsize=9)
for bar, v in zip(bars2, after_scores):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.002,
                 f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_shown, rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel('F1-Weighted Score')
axes[0].set_title('Classification: Before vs After Tuning', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 1.1)

# Improvement percentage
improvements = [(tuned_results[m] - baselines[m]['test_f1']) / baselines[m]['test_f1'] * 100
                for m in models_shown]
colors_imp = [PALETTE['mint'] if v >= 0 else PALETTE['sand'] for v in improvements]
axes[1].barh(models_shown, improvements, color=colors_imp, edgecolor='white')
axes[1].axvline(0, color='gray', lw=1.5, ls='--')
for i, v in enumerate(improvements):
    axes[1].text(v + 0.1 if v >= 0 else v - 0.1, i,
                 f'{v:+.2f}%', va='center', ha='left' if v>=0 else 'right',
                 fontsize=10, fontweight='bold')
axes[1].set_title('Relative Improvement from Tuning (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('% Change in F1-Weighted')

plt.suptitle('Hyperparameter Tuning Results — Classification Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/charts/nb3_clf_tuning.png', bbox_inches='tight', dpi=130)
plt.show()

# Best overall classifier
best_tuned_clf_name = max(tuned_results, key=tuned_results.get)
print(f"\n🏆 Best tuned classifier: {best_tuned_clf_name} (F1={tuned_results[best_tuned_clf_name]:.4f})")


In [ ]:
# ─── 1.7 LEARNING CURVES — BEST CLASSIFIER ───────────────────────────────────
best_clf_tuned = rf_best  # Use RF as it's always available

train_sizes, train_scores, val_scores = learning_curve(
    best_clf_tuned, X, y_clf, cv=skf,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='f1_weighted', n_jobs=-1
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Learning curve
train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

axes[0].plot(train_sizes, train_mean, 'o-', color=PALETTE['sky'], lw=2.5,
             markersize=7, markerfacecolor='white', markeredgewidth=2, label='Train F1')
axes[0].fill_between(train_sizes, train_mean-train_std, train_mean+train_std,
                      alpha=0.1, color=PALETTE['sky'])
axes[0].plot(train_sizes, val_mean, 's-', color=PALETTE['mint'], lw=2.5,
             markersize=7, markerfacecolor='white', markeredgewidth=2, label='CV F1')
axes[0].fill_between(train_sizes, val_mean-val_std, val_mean+val_std,
                      alpha=0.1, color=PALETTE['mint'])
axes[0].set_xlabel('Training Set Size')
axes[0].set_ylabel('F1-Weighted Score')
axes[0].set_title('Learning Curve — Best Classifier\n(Does more data help?)',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

gap = train_mean - val_mean
bias_zone  = val_mean < 0.80
var_zone   = gap > 0.10
axes[0].text(0.03, 0.08,
    f'Final gap (bias): {gap[-1]:.4f}\n'
    f'Overfit risk: {"High" if gap[-1] > 0.10 else "Low"}',
    transform=axes[0].transAxes, fontsize=9,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

# n_estimators sensitivity
n_est_range = [50, 100, 150, 200, 300, 400, 500]
n_est_scores = []
best_params_copy = rf_search.best_params_.copy()
for n in n_est_range:
    p = best_params_copy.copy()
    p['n_estimators'] = n
    rf_temp = RandomForestClassifier(**p, random_state=42, n_jobs=-1)
    cv_s = cross_val_score(rf_temp, X, y_clf, cv=3, scoring='f1_weighted', n_jobs=-1)
    n_est_scores.append(cv_s.mean())

axes[1].plot(n_est_range, n_est_scores, 'D-', color=PALETTE['sky'], lw=2.5,
             markersize=8, markerfacecolor='white', markeredgewidth=2)
axes[1].fill_between(n_est_range, [s-0.003 for s in n_est_scores],
                      [s+0.003 for s in n_est_scores], alpha=0.15, color=PALETTE['sky'])
best_n = n_est_range[np.argmax(n_est_scores)]
axes[1].axvline(best_n, color=PALETTE['sand'], ls='--', lw=2,
                label=f'Best n_estimators={best_n}')
axes[1].set_xlabel('n_estimators')
axes[1].set_ylabel('CV F1-Weighted')
axes[1].set_title('Sensitivity: n_estimators vs CV F1', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.suptitle('Learning Curves & Parameter Sensitivity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Regression Hyperparameter Tuning

In [ ]:
# ─── 2.1 BEFORE TUNING: REGRESSION BASELINES ─────────────────────────────────
print("=" * 65)
print("REGRESSION BASELINES (Default Parameters)")
print("=" * 65)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
reg_baselines = {}
reg_baseline_models = {
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'Extra Trees':       ExtraTreesRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}
if HAS_XGB:
    reg_baseline_models['XGBoost'] = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
if HAS_LGB:
    reg_baseline_models['LightGBM'] = LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)

print(f"\n{'Model':<22} {'CV R² Mean':>11} {'CV R² Std':>10} {'Test RMSE':>10} {'Test R²':>8}")
print("─" * 65)
for name, model in reg_baseline_models.items():
    cv = cross_val_score(model, X, y_reg, cv=kf, scoring='r2', n_jobs=-1)
    model.fit(X_train_r, y_train_r)
    y_pred_r = model.predict(X_test_r)
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
    r2   = r2_score(y_test_r, y_pred_r)
    reg_baselines[name] = {'cv_mean':cv.mean(),'cv_std':cv.std(),'rmse':rmse,'r2':r2}
    print(f"  {name:<20} {cv.mean():>11.4f} {cv.std():>10.4f} {rmse:>10.4f} {r2:>8.4f}")


In [ ]:
# ─── 2.2 GRADIENT BOOSTING REGRESSOR — FULL TUNING ───────────────────────────
gbr_param_dist = {
    'n_estimators':    randint(150, 500),
    'max_depth':       [3, 4, 5, 6, 7],
    'learning_rate':   loguniform(0.01, 0.3),
    'subsample':       uniform(0.6, 0.4),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf':  randint(1, 10),
    'max_features':    ['sqrt', 'log2', 0.4, 0.6, None],
    'alpha':           [0.7, 0.8, 0.9],
}

print("Tuning Gradient Boosting Regressor (n_iter=25, 5-fold KFold)...")
gbr_search = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    gbr_param_dist, n_iter=25, cv=kf,
    scoring='r2', random_state=42, n_jobs=-1, verbose=0
)
t0 = time.time()
gbr_search.fit(X, y_reg)
elapsed = time.time() - t0

gbr_best = gbr_search.best_estimator_
gbr_pred = gbr_best.predict(X_test_r)
gbr_r2   = r2_score(y_test_r, gbr_pred)
gbr_rmse = np.sqrt(mean_squared_error(y_test_r, gbr_pred))
gbr_mae  = mean_absolute_error(y_test_r, gbr_pred)

print(f"\n✓ Search complete in {elapsed:.1f}s ({25*5} fits)")
print(f"  Best CV R²          : {gbr_search.best_score_:.4f}")
print(f"  Best Parameters     :")
for k, v in gbr_search.best_params_.items():
    print(f"    {k:<22}: {v:.4f}" if isinstance(v,float) else f"    {k:<22}: {v}")
print(f"\n  Regression Metrics (Test Set):")
print(f"    R² Score  : {gbr_r2:.4f}  (baseline: {reg_baselines['Gradient Boosting']['r2']:.4f})")
print(f"    RMSE      : {gbr_rmse:.4f}  (baseline: {reg_baselines['Gradient Boosting']['rmse']:.4f})")
print(f"    MAE       : {gbr_mae:.4f}")
print(f"    Improvement R²: {gbr_r2 - reg_baselines['Gradient Boosting']['r2']:+.4f}")


In [ ]:
# ─── 2.3 RANDOM FOREST REGRESSOR TUNING ─────────────────────────────────────
rfr_param_dist = {
    'n_estimators':    randint(100, 400),
    'max_depth':       [8, 12, 15, 20, None],
    'min_samples_split': randint(2, 15),
    'min_samples_leaf':  randint(1, 8),
    'max_features':    ['sqrt', 'log2', 0.3, 0.5, None],
    'bootstrap':       [True, False],
    'oob_score':       [True, False],
}
print("Tuning Random Forest Regressor (n_iter=20)...")
rfr_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rfr_param_dist, n_iter=20, cv=kf,
    scoring='r2', random_state=42, n_jobs=-1
)
rfr_search.fit(X, y_reg)
rfr_best = rfr_search.best_estimator_
rfr_pred = rfr_best.predict(X_test_r)
rfr_r2   = r2_score(y_test_r, rfr_pred)
rfr_rmse = np.sqrt(mean_squared_error(y_test_r, rfr_pred))
print(f"  Best CV R²: {rfr_search.best_score_:.4f}")
print(f"  Test R²: {rfr_r2:.4f}  RMSE: {rfr_rmse:.4f}")
print(f"  Best params: {rfr_search.best_params_}")


In [ ]:
# ─── 2.4 REGRESSION BEFORE vs AFTER + RESIDUAL ANALYSIS ─────────────────────
tuned_regressors = {
    'Gradient Boosting': (gbr_r2, gbr_rmse, gbr_best),
    'Random Forest':     (rfr_r2, rfr_rmse, rfr_best),
}

fig = plt.figure(figsize=(20, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

# R² comparison
ax1 = fig.add_subplot(gs[0, 0])
models_r = list(tuned_regressors.keys())
before_r2 = [reg_baselines[m]['r2'] for m in models_r]
after_r2  = [tuned_regressors[m][0] for m in models_r]
x = np.arange(len(models_r))
w = 0.38
ax1.bar(x-w/2, before_r2, w, label='Before', color=PALETTE['sand'], alpha=0.85, edgecolor='white')
ax1.bar(x+w/2, after_r2,  w, label='After',  color=PALETTE['sky'],  alpha=0.85, edgecolor='white')
for i,(b,a) in enumerate(zip(before_r2,after_r2)):
    ax1.text(i-w/2, b+0.002, f'{b:.3f}', ha='center', fontsize=9)
    ax1.text(i+w/2, a+0.002, f'{a:.3f}', ha='center', fontsize=9, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(models_r, fontsize=9)
ax1.set_title('R² Before vs After Tuning', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9); ax1.set_ylabel('R² Score')

# Pred vs Actual
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(y_test_r[:3000], gbr_pred[:3000], alpha=0.25, s=10,
            color=PALETTE['sky'], edgecolors='none')
mn, mx = y_test_r.min(), y_test_r.max()
ax2.plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect fit')
ax2.set_xlabel('Actual Rating'); ax2.set_ylabel('Predicted Rating')
ax2.set_title(f'Pred vs Actual — GBR (R²={gbr_r2:.4f})', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)

# Residuals by bucket
ax3 = fig.add_subplot(gs[0, 2])
residuals = y_test_r - gbr_pred
rating_buckets = pd.cut(y_test_r, bins=[3.0,3.5,4.0,4.5,5.01],
                         labels=['3.0-3.5','3.5-4.0','4.0-4.5','4.5-5.0'])
res_by_bucket = pd.DataFrame({'residual':residuals,'bucket':rating_buckets}).groupby('bucket',observed=True)['residual']
means = res_by_bucket.mean(); stds = res_by_bucket.std()
ax3.bar(range(len(means)), means.values, yerr=stds.values,
        color=CHART_COLORS[:4], capsize=4, edgecolor='white')
ax3.set_xticks(range(len(means))); ax3.set_xticklabels(means.index, fontsize=9)
ax3.axhline(0, color='red', lw=1.5, ls='--')
ax3.set_title('Residuals by Rating Bucket (Mean ± Std)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Residual')

# Learning curve for best regressor
ax4 = fig.add_subplot(gs[1, :2])
train_sizes_r, train_scores_r, val_scores_r = learning_curve(
    gbr_best, X, y_reg, cv=kf, scoring='r2',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1
)
tm = train_scores_r.mean(axis=1); ts = train_scores_r.std(axis=1)
vm = val_scores_r.mean(axis=1);   vs = val_scores_r.std(axis=1)
ax4.plot(train_sizes_r, tm, 'o-', color=PALETTE['sky'], lw=2.5, label='Train R²')
ax4.fill_between(train_sizes_r, tm-ts, tm+ts, alpha=0.1, color=PALETTE['sky'])
ax4.plot(train_sizes_r, vm, 's-', color=PALETTE['mint'], lw=2.5, label='CV R²')
ax4.fill_between(train_sizes_r, vm-vs, vm+vs, alpha=0.1, color=PALETTE['mint'])
ax4.set_xlabel('Training Set Size'); ax4.set_ylabel('R² Score')
ax4.set_title('Learning Curve — GBR (Tuned)', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10); ax4.grid(alpha=0.3)

# Feature importance
ax5 = fig.add_subplot(gs[1, 2])
fi = pd.Series(gbr_best.feature_importances_, index=FEATURE_COLS).nlargest(12)
ax5.barh(range(len(fi)), fi.values[::-1],
         color=plt.cm.Blues_r(np.linspace(0.2,0.9,len(fi)))[::-1])
ax5.set_yticks(range(len(fi)))
ax5.set_yticklabels(fi.index[::-1], fontsize=9)
ax5.set_title('GBR Feature Importances (Top 12)', fontsize=12, fontweight='bold')
ax5.set_xlabel('Importance')
for i, v in enumerate(fi.values[::-1]):
    ax5.text(v+0.0003, i, f'{v:.4f}', va='center', fontsize=8)

plt.suptitle('Regression Tuning Analysis', fontsize=13, fontweight='bold', y=1.01)
plt.savefig('../static/charts/nb3_reg_tuning.png', bbox_inches='tight', dpi=130)
plt.show()


## 3. TF-IDF & Content Recommender Tuning

In [ ]:
# ─── 3.1 TF-IDF HYPERPARAMETER GRID SEARCH ───────────────────────────────────
# Evaluate TF-IDF configurations against retrieval quality metrics
corpus = catalog['combined_features'].fillna('').tolist()
n_corpus = len(corpus)

tfidf_configs = [
    {'max_features': 1000,  'ngram_range': (1,1), 'min_df': 1, 'max_df': 1.0,  'sublinear_tf': False},
    {'max_features': 2000,  'ngram_range': (1,2), 'min_df': 1, 'max_df': 0.95, 'sublinear_tf': False},
    {'max_features': 3000,  'ngram_range': (1,2), 'min_df': 1, 'max_df': 0.95, 'sublinear_tf': True},
    {'max_features': 5000,  'ngram_range': (1,2), 'min_df': 1, 'max_df': 0.95, 'sublinear_tf': True},
    {'max_features': 5000,  'ngram_range': (1,3), 'min_df': 1, 'max_df': 0.90, 'sublinear_tf': True},
    {'max_features': 8000,  'ngram_range': (1,2), 'min_df': 1, 'max_df': 0.95, 'sublinear_tf': True},
    {'max_features': 10000, 'ngram_range': (1,2), 'min_df': 1, 'max_df': 0.95, 'sublinear_tf': True},
]

print("=" * 85)
print(f"{'Config':<8} {'Features':>8} {'NGram':>8} {'SubLin':>7} {'Vocab':>7} "
      f"{'AvgSim@10':>10} {'Coverage':>9} {'Score':>8}")
print("─" * 85)

best_tfidf_score = -1
best_tfidf_cfg   = None
tfidf_results    = []

# Use ground-truth category queries to evaluate retrieval precision
query_tests = [
    ('Natural Scenery Spring Autumn mountain', 'Natural Scenery'),
    ('Historical Culture ancient dynasty heritage', 'Historical Culture'),
    ('Theme Park entertainment family summer', 'Theme Park'),
    ('Religious Culture temple shrine spiritual', 'Religious Culture'),
]

for i, cfg in enumerate(tfidf_configs):
    tfidf_tmp = TfidfVectorizer(**cfg)
    mat = tfidf_tmp.fit_transform(corpus)

    # Metric 1: Average intra-category similarity (precision proxy)
    cat_sims = []
    for query_text, true_cat in query_tests:
        qvec = tfidf_tmp.transform([query_text])
        sims = cosine_similarity(qvec, mat)[0]
        top10_idx = sims.argsort()[-10:][::-1]
        top10_cats = [catalog.iloc[j]['category'] for j in top10_idx if j < len(catalog)]
        precision_at_10 = sum(1 for c in top10_cats if c == true_cat) / len(top10_cats)
        cat_sims.append(precision_at_10)
    avg_precision = np.mean(cat_sims)

    # Metric 2: Coverage (unique attractions retrievable in top-10 across all queries)
    all_retrieved = set()
    for query_text, _ in query_tests:
        qvec = tfidf_tmp.transform([query_text])
        sims = cosine_similarity(qvec, mat)[0]
        all_retrieved.update(sims.argsort()[-10:][::-1])
    coverage = len(all_retrieved) / len(catalog)

    # Composite score
    composite = 0.6 * avg_precision + 0.4 * coverage
    tfidf_results.append({**cfg, 'vocab_size': mat.shape[1], 'precision': avg_precision,
                           'coverage': coverage, 'composite': composite})

    if composite > best_tfidf_score:
        best_tfidf_score = composite
        best_tfidf_cfg   = cfg

    print(f"  Cfg {i+1:<3} {cfg['max_features']:>8,} {str(cfg['ngram_range']):>8} "
          f"{'Y' if cfg['sublinear_tf'] else 'N':>7} {mat.shape[1]:>7,} "
          f"{avg_precision:>10.4f} {coverage:>9.4f} {composite:>8.4f}"
          + (" ← BEST" if composite == best_tfidf_score else ""))
print("─" * 85)
print(f"\n✓ Best TF-IDF config: {best_tfidf_cfg}")
print(f"  Composite score   : {best_tfidf_score:.4f}")


In [ ]:
# ─── 3.2 TF-IDF RESULTS VISUALISATION ────────────────────────────────────────
tfidf_df = pd.DataFrame(tfidf_results)
tfidf_df['config_label'] = [f"MF={r['max_features']}\nNG={r['ngram_range']}\nSL={'Y' if r['sublinear_tf'] else 'N'}"
                             for _, r in tfidf_df.iterrows()]

fig, axes = plt.subplots(1, 3, figsize=(19, 6))

axes[0].bar(range(len(tfidf_df)), tfidf_df['precision'],
            color=[PALETTE['sky'] if v == tfidf_df['precision'].max() else PALETTE['sky_l']
                   for v in tfidf_df['precision']], edgecolor='white')
axes[0].set_xticks(range(len(tfidf_df)))
axes[0].set_xticklabels(tfidf_df['config_label'], fontsize=7.5)
axes[0].set_title('Precision@10 (Category Match Rate)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Precision@10')
for i, v in enumerate(tfidf_df['precision']): axes[0].text(i, v+0.002, f'{v:.3f}', ha='center', fontsize=8)

axes[1].bar(range(len(tfidf_df)), tfidf_df['coverage'],
            color=[PALETTE['mint'] if v == tfidf_df['coverage'].max() else '#d1fae5'
                   for v in tfidf_df['coverage']], edgecolor='white')
axes[1].set_xticks(range(len(tfidf_df)))
axes[1].set_xticklabels(tfidf_df['config_label'], fontsize=7.5)
axes[1].set_title('Coverage (Catalog Reach)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Coverage %')
for i, v in enumerate(tfidf_df['coverage']): axes[1].text(i, v+0.002, f'{v:.3f}', ha='center', fontsize=8)

colors_comp = [PALETTE['sand'] if v == tfidf_df['composite'].max() else PALETTE['snow']
               for v in tfidf_df['composite']]
bars = axes[2].bar(range(len(tfidf_df)), tfidf_df['composite'],
                   color=colors_comp, edgecolor='white')
axes[2].set_xticks(range(len(tfidf_df)))
axes[2].set_xticklabels(tfidf_df['config_label'], fontsize=7.5)
axes[2].set_title('Composite Score (0.6×Prec + 0.4×Cov)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Composite Score')
for i, v in enumerate(tfidf_df['composite']):
    weight = 'bold' if v == tfidf_df['composite'].max() else 'normal'
    axes[2].text(i, v+0.001, f'{v:.3f}', ha='center', fontsize=8, fontweight=weight)

plt.suptitle('TF-IDF Configuration Grid Search Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/charts/nb3_tfidf_tuning.png', bbox_inches='tight', dpi=130)
plt.show()


## 4. Hybrid Recommender Weight Tuning

In [ ]:
# ─── 4.1 HYBRID WEIGHT ABLATION STUDY ────────────────────────────────────────
# Build best content recommender first
best_tfidf = TfidfVectorizer(**best_tfidf_cfg)
best_tfidf.fit(corpus)

from src.recommender import ContentBasedRecommender, HybridRecommender
content_rec = ContentBasedRecommender()
content_rec.fit(catalog, tfidf_vectorizer=best_tfidf)

def evaluate_hybrid(weights, catalog, content_rec, n_queries=40):
    """
    Evaluate hybrid recommender across diverse queries.
    Returns dict with quality, diversity, novelty, coverage metrics.
    """
    hr = HybridRecommender(weights=weights)
    hr.fit(catalog, content_rec)

    top_pop_attractions = catalog.nlargest(int(len(catalog)*0.2),'popularity_score')['attraction_name'].tolist()
    test_params = [
        {'category': c, 'season': s, 'n': 10}
        for c in catalog['category'].value_counts().head(5).index
        for s in ['Spring','Autumn']
    ]

    quality_scores, diversity_scores, novelty_scores, all_recs = [], [], [], set()
    for tp in test_params:
        try:
            r = hr.recommend(**tp)
            if r.empty: continue
            quality_scores.append(r['avg_rating'].mean())
            cats = r['category'].tolist()
            diversity_scores.append(len(set(cats)) / len(cats))
            novelty_scores.append((~r['attraction_name'].isin(top_pop_attractions)).mean())
            all_recs.update(r['attraction_name'].tolist())
        except:
            pass

    return {
        'quality':   np.mean(quality_scores)   if quality_scores   else 0,
        'diversity': np.mean(diversity_scores) if diversity_scores else 0,
        'novelty':   np.mean(novelty_scores)   if novelty_scores   else 0,
        'coverage':  len(all_recs) / len(catalog),
    }

# Weight configurations to test (rows must sum to 1.0)
weight_configs = {
    'Similarity-Heavy':   {'similarity':0.45,'popularity':0.20,'rating':0.15,'budget':0.10,'season':0.05,'category':0.05},
    'Popularity-Heavy':   {'similarity':0.20,'popularity':0.40,'rating':0.20,'budget':0.07,'season':0.07,'category':0.06},
    'Rating-Heavy':       {'similarity':0.25,'popularity':0.20,'rating':0.35,'budget':0.08,'season':0.07,'category':0.05},
    'Balanced (Default)': {'similarity':0.30,'popularity':0.25,'rating':0.20,'budget':0.10,'season':0.10,'category':0.05},
    'Season-Aware':       {'similarity':0.25,'popularity':0.20,'rating':0.15,'budget':0.10,'season':0.25,'category':0.05},
    'Budget-Sensitive':   {'similarity':0.25,'popularity':0.20,'rating':0.15,'budget':0.25,'season':0.10,'category':0.05},
    'Diversity-Focused':  {'similarity':0.20,'popularity':0.15,'rating':0.20,'budget':0.10,'season':0.10,'category':0.25},
}

print("=" * 90)
print(f"{'Config':<22} {'Quality':>9} {'Diversity':>10} {'Novelty':>8} {'Coverage':>9} {'Composite':>10}")
print("─" * 90)

ablation_results = {}
for name, weights in weight_configs.items():
    metrics = evaluate_hybrid(weights, catalog, content_rec)
    composite = (0.40*metrics['quality']/5.0 + 0.25*metrics['diversity'] +
                 0.20*metrics['novelty'] + 0.15*metrics['coverage'])
    ablation_results[name] = {**metrics, 'composite': composite, 'weights': weights}
    print(f"  {name:<20} {metrics['quality']:>9.4f} {metrics['diversity']:>10.4f} "
          f"{metrics['novelty']:>8.4f} {metrics['coverage']:>9.4f} {composite:>10.4f}")
print("─" * 90)

best_config_name = max(ablation_results, key=lambda k: ablation_results[k]['composite'])
best_weights = ablation_results[best_config_name]['weights']
print(f"\n🏆 Best weight configuration: {best_config_name}")
print(f"   Weights: {best_weights}")
print(f"   Composite score: {ablation_results[best_config_name]['composite']:.4f}")


In [ ]:
# ─── 4.2 WEIGHT ABLATION VISUALISATION ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

config_names = list(ablation_results.keys())
metrics_to_plot = [
    ('quality',   'Recommendation Quality (Avg Rating)', axes[0,0], PALETTE['sky']),
    ('diversity', 'Result Diversity',                    axes[0,1], PALETTE['mint']),
    ('novelty',   'Novelty Score',                       axes[1,0], PALETTE['sand']),
    ('composite', 'Composite Score (Production Metric)', axes[1,1], '#8b5cf6'),
]

for metric, title, ax, color in metrics_to_plot:
    vals = [ablation_results[c][metric] for c in config_names]
    bar_colors = [color if c == best_config_name else f'{color}55'
                  if len(color) == 7 else color[:7]+'55' for c in config_names]
    bars = ax.bar(range(len(config_names)), vals, color=bar_colors, edgecolor='white')
    ax.set_xticks(range(len(config_names)))
    ax.set_xticklabels(config_names, rotation=25, ha='right', fontsize=8.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.001,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8.5,
                fontweight='bold' if config_names[bars.index(bar)] == best_config_name else 'normal')
    ax.axvline(config_names.index(best_config_name), color='gray',
               lw=1.5, ls=':', alpha=0.6, label='Best config')
    ax.legend(fontsize=9)

plt.suptitle('Hybrid Weight Ablation Study\n'
             f'Best configuration: "{best_config_name}"',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../static/charts/nb3_weight_ablation.png', bbox_inches='tight', dpi=130)
plt.show()


In [ ]:
# ─── 4.3 WEIGHT SENSITIVITY ANALYSIS ─────────────────────────────────────────
# Single-parameter sensitivity: vary each weight while holding others constant
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flat

weight_keys = ['similarity', 'popularity', 'rating', 'budget', 'season', 'category']
sweep_values = np.linspace(0.05, 0.50, 10)

for ax, key in zip(axes, weight_keys):
    composite_vals = []
    for val in sweep_values:
        # Distribute remaining weight evenly
        w = best_weights.copy()
        remaining = 1.0 - val
        other_keys = [k for k in w if k != key]
        per_other = remaining / len(other_keys)
        w[key] = val
        for k in other_keys: w[k] = per_other
        try:
            m = evaluate_hybrid(w, catalog, content_rec)
            comp = (0.40*m['quality']/5.0 + 0.25*m['diversity'] +
                    0.20*m['novelty'] + 0.15*m['coverage'])
        except:
            comp = 0
        composite_vals.append(comp)

    ax.plot(sweep_values, composite_vals, 'o-', lw=2.5,
            color=PALETTE['sky'], markersize=7, markerfacecolor='white', markeredgewidth=2)
    ax.fill_between(sweep_values, composite_vals, alpha=0.12, color=PALETTE['sky'])
    ax.axvline(best_weights[key], color=PALETTE['sand'], ls='--', lw=2,
               label=f'Current={best_weights[key]:.2f}')
    ax.set_xlabel(f'{key} weight', fontsize=10)
    ax.set_ylabel('Composite Score', fontsize=10)
    ax.set_title(f'Sensitivity: {key}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    optimal_val = sweep_values[np.argmax(composite_vals)]
    ax.text(0.03, 0.93, f'Optimal≈{optimal_val:.2f}', transform=ax.transAxes,
            fontsize=9, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Weight Sensitivity Analysis — How Each Weight Affects Overall Performance',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/charts/nb3_weight_sensitivity.png', bbox_inches='tight', dpi=130)
plt.show()


## 5. Save All Best Parameters

In [ ]:
# ─── 5. SAVE BEST TUNED PARAMETERS TO JSON ───────────────────────────────────
import json, os

# Collect all best parameters
tuned_params = {
    'classification': {
        'best_model': 'Random Forest',
        'best_params': {k: (int(v) if isinstance(v, np.integer) else
                            float(v) if isinstance(v, np.floating) else v)
                        for k, v in rf_search.best_params_.items()},
        'best_cv_f1': round(float(rf_search.best_score_), 6),
        'best_test_f1': round(float(rf_test_f1), 6),
        'baseline_test_f1': round(float(baselines['Random Forest']['test_f1']), 6),
        'improvement': round(float(rf_test_f1 - baselines['Random Forest']['test_f1']), 6),
    },
    'regression': {
        'best_model': 'Gradient Boosting',
        'best_params': {k: (int(v) if isinstance(v, np.integer) else
                            float(v) if isinstance(v, np.floating) else v)
                        for k, v in gbr_search.best_params_.items()},
        'best_cv_r2': round(float(gbr_search.best_score_), 6),
        'best_test_r2': round(float(gbr_r2), 6),
        'best_test_rmse': round(float(gbr_rmse), 6),
        'best_test_mae': round(float(gbr_mae), 6),
        'baseline_test_r2': round(float(reg_baselines['Gradient Boosting']['r2']), 6),
        'improvement': round(float(gbr_r2 - reg_baselines['Gradient Boosting']['r2']), 6),
    },
    'tfidf': {
        'best_config': {k: (list(v) if isinstance(v, tuple) else v)
                        for k, v in best_tfidf_cfg.items()},
        'best_composite_score': round(float(best_tfidf_score), 6),
        'all_configs_ranked': [
            {k: (list(v) if isinstance(v, tuple) else
                 float(v) if isinstance(v, np.floating) else
                 int(v) if isinstance(v, np.integer) else v)
             for k, v in sorted(r.items(), key=str)}
            for r in sorted(tfidf_results, key=lambda x: x['composite'], reverse=True)
        ],
    },
    'hybrid_weights': {
        'best_config_name': best_config_name,
        'best_weights': best_weights,
        'best_composite': round(float(ablation_results[best_config_name]['composite']), 6),
        'all_configs': {
            name: {k: (float(v) if isinstance(v, float) else v)
                   for k, v in {**res, 'weights': res['weights']}.items()
                   if k != 'weights'}
            for name, res in ablation_results.items()
        },
    },
    'metadata': {
        'tuning_timestamp': __import__('datetime').datetime.now().isoformat(),
        'n_classification_fits': 30 * 5,
        'n_regression_fits': 25 * 5,
        'n_tfidf_configs': len(tfidf_configs),
        'n_weight_configs': len(weight_configs),
    }
}

# Load existing metadata and update
meta_path = '../models/model_metadata.json'
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
else:
    meta = {}

meta['tuned_params'] = tuned_params
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print("✓ All tuned parameters saved to models/model_metadata.json")
print()
print("=" * 65)
print("TUNING SUMMARY")
print("=" * 65)
print(f"  Classification Best F1  : {tuned_params['classification']['best_test_f1']:.4f}")
print(f"  Classification Baseline : {tuned_params['classification']['baseline_test_f1']:.4f}")
print(f"  Improvement             : {tuned_params['classification']['improvement']:+.4f}")
print()
print(f"  Regression Best R²      : {tuned_params['regression']['best_test_r2']:.4f}")
print(f"  Regression Best RMSE    : {tuned_params['regression']['best_test_rmse']:.4f}")
print(f"  Regression Baseline R²  : {tuned_params['regression']['baseline_test_r2']:.4f}")
print(f"  Improvement             : {tuned_params['regression']['improvement']:+.4f}")
print()
print(f"  Best TF-IDF Config      : {best_tfidf_cfg}")
print(f"  Best Hybrid Config      : {best_config_name}")
print(f"  Best Hybrid Weights     : {best_weights}")
print("=" * 65)
print("\n→ Proceed to Notebook 04 for final evaluation and export")
